# Atmospheric-component baseline

The target is daily mean PM2.5 concentration for a `Place_ID` on a given date. PM2.5 means fine particulate matter with diameter smaller than 2.5 micrometers; air-quality datasets usually report this concentration in `ug/m^3` (micrograms per cubic meter of air). In this Zindi dataset, the target is measured by ground-based sensors, while the model inputs are weather fields plus Sentinel-5P Level 3 satellite retrievals of atmospheric gases/aerosols.

`target_min`, `target_max`, `target_variance`, and `target_count` are not usable predictors for test-time modeling because they are same-day summaries from the ground sensor target readings. Treat them as leakage.

In [31]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline

clean_train = pd.read_csv("data/clean_train.csv")
clean_val = pd.read_csv("data/clean_val.csv")

print(clean_train.shape, clean_val.shape)
clean_train["target"].describe()


(24546, 71) (6011, 71)


count    24546.000000
mean        62.030995
std         47.960782
min          1.000000
25%         26.000000
50%         51.000000
75%         80.000000
max        815.000000
Name: target, dtype: float64

## Density-only component features

This feature set keeps only atmospheric density measurements: column, slant-column, tropospheric, and stratospheric densities. It excludes aerosol index features, AMF correction factors, sensor/solar viewing geometry, clouds, location IDs, dates, weather variables, and target-derived leakage.

In [32]:
component_features = [
    "L3_NO2_NO2_column_number_density",
    "L3_NO2_NO2_slant_column_number_density",
    "L3_NO2_stratospheric_NO2_column_number_density",
    "L3_NO2_tropospheric_NO2_column_number_density",
    "L3_O3_O3_column_number_density",
    "L3_CO_CO_column_number_density",
    "L3_CO_H2O_column_number_density",
    "L3_HCHO_HCHO_slant_column_number_density",
    "L3_HCHO_tropospheric_HCHO_column_number_density",
    "L3_SO2_SO2_column_number_density",
    "L3_SO2_SO2_slant_column_number_density",
]

component_features


['L3_NO2_NO2_column_number_density',
 'L3_NO2_NO2_slant_column_number_density',
 'L3_NO2_stratospheric_NO2_column_number_density',
 'L3_NO2_tropospheric_NO2_column_number_density',
 'L3_O3_O3_column_number_density',
 'L3_CO_CO_column_number_density',
 'L3_CO_H2O_column_number_density',
 'L3_HCHO_HCHO_slant_column_number_density',
 'L3_HCHO_tropospheric_HCHO_column_number_density',
 'L3_SO2_SO2_column_number_density',
 'L3_SO2_SO2_slant_column_number_density']

## Correlation check

`corr` stores the absolute Pearson correlation between each component feature and `target`, sorted from strongest to weakest. It is a quick linear relationship check: values closer to 1 mean a stronger straight-line relationship with PM2.5; values near 0 mean little linear relationship. It does not prove causation, and it can miss nonlinear effects.

In [33]:
corr = (
    clean_train[component_features + ["target"]]
    .corr(numeric_only=True)["target"]
    .drop("target")
    .abs()
    .sort_values(ascending=False)
)

corr


L3_CO_CO_column_number_density                     0.346478
L3_HCHO_tropospheric_HCHO_column_number_density    0.317968
L3_HCHO_HCHO_slant_column_number_density           0.298706
L3_NO2_NO2_column_number_density                   0.298639
L3_NO2_tropospheric_NO2_column_number_density      0.287960
L3_NO2_NO2_slant_column_number_density             0.284075
L3_O3_O3_column_number_density                     0.108804
L3_SO2_SO2_slant_column_number_density             0.068865
L3_SO2_SO2_column_number_density                   0.063447
L3_NO2_stratospheric_NO2_column_number_density     0.034757
L3_CO_H2O_column_number_density                    0.017226
Name: target, dtype: float64

## Baseline model

The split comes from `eda_cleaning.ipynb`: places are split into train/validation groups, then missing values and outliers are handled using training-only statistics. This is a useful validation setup because it tests generalization to held-out places.

This baseline is strictly first-degree linear regression: each feature enters only to the power of 1. There are no polynomial, squared, or interaction terms.

In [34]:
X_train = clean_train[component_features]
y_train = clean_train["target"]
X_val = clean_val[component_features]
y_val = clean_val["target"]

baseline = make_pipeline(
    SimpleImputer(strategy="median"),
    LinearRegression(),
)

baseline.fit(X_train, y_train)
val_pred = baseline.predict(X_val)

metrics = {
    "rmse": mean_squared_error(y_val, val_pred, squared=False),
    "mae": mean_absolute_error(y_val, val_pred),
    "r2": r2_score(y_val, val_pred),
}

metrics

{'rmse': 37.22883903115352,
 'mae': 28.010956493845075,
 'r2': 0.20985538944826665}

In [35]:
linear_model = baseline.named_steps["linearregression"]
coefficients = (
    pd.Series(linear_model.coef_, index=component_features)
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .to_frame("coefficient")
)

coefficients

,coefficient
L3_NO2_stratospheric_NO2_column_number_density,-635357.438415
L3_HCHO_tropospheric_HCHO_column_number_density,254640.370369
L3_NO2_NO2_slant_column_number_density,238093.422749
L3_NO2_tropospheric_NO2_column_number_density,-180558.571394
L3_NO2_NO2_column_number_density,114374.386064
L3_HCHO_HCHO_slant_column_number_density,-101990.315951
L3_SO2_SO2_slant_column_number_density,46192.643027
L3_SO2_SO2_column_number_density,-8492.566759
L3_CO_CO_column_number_density,1940.450772
L3_O3_O3_column_number_density,-331.216013
